In [1]:
%load_ext autoreload
%autoreload 2
import eval
import os
import glob
import pickle

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


/data/cb/mihirb14/projects/BoltzDesign1/utils/geometry.py:8: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(False)
/data/cb/mihirb14/projects/BoltzDesign1/utils/geometry.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(False)
/data/cb/mihirb14/projects/BoltzDesign1/utils/geometry.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(False)


In [2]:
# out_dir = f"./out/onemotif_twostates_pos/"
out_dir = f"/data/cb/mihirb14/projects/BoltzDesign1/out/test/zinc/onemotif_twostates_pos"

motif = "3ixt"
motif_pdb = f"/data/cb/mihirb14/projects/BoltzDesign1/motifs/{motif}.pdb"
motif_out_dir = os.path.join(out_dir,motif)

In [3]:


def motif_results(motif_out_dir, motif_pdb, motif):
    rows = []
    for design_dir in sorted(glob.glob(os.path.join(motif_out_dir, "design*"))):
        design_name = os.path.basename(design_dir)

        # load motif mask
        with open(os.path.join(design_dir, f"{motif}_spec.pkl"), "rb") as f:
            motif_mask = pickle.load(f)["motif_mask"]

        for state in [0, 1]:
            with open(os.path.join(design_dir, f"state{state}.pkl"), "rb") as f:
                outdict = pickle.load(f)

            for sample_idx in range(5):  # assume 5 samples per state
                pdb_file = os.path.join(design_dir, f"state{state}_sample{sample_idx}.pdb")
                if not os.path.exists(pdb_file):
                    continue

                rmsd = eval.motifRMSD(motif_pdb, pdb_file, motif_mask)

                rows.append({
                    "design": design_name,
                    "state": state,
                    "sample": sample_idx,
                    "motifrmsd": rmsd.item() if hasattr(rmsd, "item") else float(rmsd),
                    "plddt": outdict["plddt"].cpu().numpy()[sample_idx].mean(),
                    "ptm": outdict["ptm"].cpu().numpy()[sample_idx],
                })
                
    full = pd.DataFrame(rows)

    agg = full.groupby(["design", "state"]).agg(
        motifRMSD_mean=("motifrmsd", "mean"),
        motifRMSD_std=("motifrmsd", "std"),
        plddt=("plddt", "mean"),
        ptm=("ptm", "mean")
    ).reset_index()

    agg = agg.pivot(index="design", columns="state").reset_index()
    agg.columns = ["_".join(map(str, col)).rstrip("_") for col in agg.columns.to_flat_index()]
    agg = agg.rename(columns=lambda c: c.replace("_0", "_unbound").replace("_1", "_bound"))

    return full, agg



df_full,df_agg = motif_results(motif_out_dir,motif_pdb, motif)
df_agg

,design,motifRMSD_mean_unbound,motifRMSD_mean_bound,motifRMSD_std_unbound,motifRMSD_std_bound,plddt_unbound,plddt_bound,ptm_unbound,ptm_bound
0,design0,4.512683,3.652910,1.397996,0.128000,0.670658,0.698836,0.279670,0.520907
1,design1,7.613704,2.247110,3.134679,0.704863,0.600609,0.524029,0.175832,0.329739
2,design10,3.925962,3.250747,2.477956,1.559058,0.644481,0.651648,0.291849,0.375208
3,design11,3.741401,3.497411,0.906390,0.368015,0.571051,0.718480,0.275792,0.665124
4,design12,7.043576,0.831177,2.381233,0.051728,0.605420,0.800228,0.252382,0.749492
...,...,...,...,...,...,...,...,...,...
95,design95,5.154456,0.782321,2.404585,0.041088,0.593664,0.751546,0.261528,0.750319
96,design96,8.665702,3.149907,0.210581,0.094190,0.663993,0.715778,0.414841,0.659128
97,design97,3.336979,1.661099,2.794116,0.833067,0.671656,0.636786,0.244261,0.370350
98,design98,4.460093,3.418604,2.489508,0.046039,0.734243,0.607880,0.354154,0.380716


In [4]:
len(df_agg[(df_agg["motifRMSD_mean_unbound"]>1.0) & (df_agg["motifRMSD_mean_bound"]<=1.0)]) # condition for one motif two states positive allostery

17

In [5]:


def plot_scatter_motifrmsd(df, motif):
    xrange = [0, 10]
    yrange = [0, 10]

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("", "with std ± 1")
    )

    fig.add_trace(
        go.Scatter(
            x=df["motifRMSD_mean_unbound"],
            y=df["motifRMSD_mean_bound"],
            mode="markers",
            marker=dict(
                color=df["plddt_unbound"],
                colorscale="sunsetdark",
                showscale=True,
                colorbar=dict(title="pLDDT",outlinewidth=0)
            ),
            text=df["design"],
            name="Designs"
        ),
        row=1, col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=df["motifRMSD_mean_unbound"],
            y=df["motifRMSD_mean_bound"],
            mode="markers",
            marker=dict(
                color=df["plddt_unbound"],
                colorscale="sunsetdark",
                showscale=False
            ),
            text=df["design"],
            error_x=dict(array=df["motifRMSD_std_unbound"], color="gray", thickness=1),
            error_y=dict(array=df["motifRMSD_std_bound"], color="gray", thickness=1),
            name="Designs (err)"
        ),
        row=1, col=2
    )

    for c in [1, 2]:
        fig.add_shape(
            type="line", x0=0, y0=0, x1=10, y1=10,
            line=dict(color="lightgray", dash="dash"),
            row=1, col=c
        )

    fig.update_xaxes(range=xrange, title="Unbound motif RMSD", row=1, col=1)
    fig.update_yaxes(range=yrange, title="Bound motif RMSD", row=1, col=1)
    fig.update_xaxes(range=xrange, title="Unbound motif RMSD", row=1, col=2)
    fig.update_yaxes(range=yrange, title="Bound motif RMSD", row=1, col=2)

    fig.update_layout(
        width=1200, height=600,
        title=f"({motif}) Unbound vs Bound motif RMSD",
        yaxis_scaleanchor="x",
        yaxis2_scaleanchor="x2",
        showlegend=False
    )

    return fig


plot_scatter_motifrmsd(df_agg, motif)


### plot all (onemotif_twostates_neg)

In [13]:
successes_neg = []
out_dir = f"./out/onemotif_twostates_neg/"

for motif in sorted(os.listdir(out_dir)):
    motif_out_dir = os.path.join(out_dir,motif)
    motif_pdb = f"/data/cb/mihirb14/projects/BoltzDesign1/motifs/{motif}.pdb"
    full, agg = motif_results(motif_out_dir, motif_pdb, motif)
    
    n_success = len(agg[(agg["motifRMSD_mean_unbound"] <= 1.0) & (agg["motifRMSD_mean_bound"] > 1.0)])# condition for one motif two states negative allostery
    successes_neg.append({"motif": motif, "task": "onemotif_twostates_neg", "success_count": n_success})
    plot_scatter_motifrmsd(agg, motif).show()
    
df_neg = pd.DataFrame(successes_neg)


In [14]:
df_neg

,motif,task,success_count
0,1bcf,onemotif_twostates_neg,0
1,1prw,onemotif_twostates_neg,6
2,1qjg,onemotif_twostates_neg,5
3,1ycr,onemotif_twostates_neg,32
4,2kl8,onemotif_twostates_neg,3
5,3ixt,onemotif_twostates_neg,15
6,4jhw,onemotif_twostates_neg,0
7,4zyp,onemotif_twostates_neg,7
8,5ius,onemotif_twostates_neg,0
9,5tpn,onemotif_twostates_neg,0


### plot all (onemotif_twostates_pos)

In [16]:
out_dir = f"./out/onemotif_twostates_pos/"
successes_pos = []

for motif in sorted(os.listdir(out_dir)):
    motif_out_dir = os.path.join(out_dir,motif)
    motif_pdb = f"/data/cb/mihirb14/projects/BoltzDesign1/motifs/{motif}.pdb"
    full, agg = motif_results(motif_out_dir, motif_pdb, motif)
    n_success = len(agg[(agg["motifRMSD_mean_unbound"] > 1.0) & (agg["motifRMSD_mean_bound"] <= 1.0)])# condition for one motif two states positive allostery
    successes_pos.append({"motif": motif, "task": "onemotif_twostates_neg", "success_count": n_success})
    plot_scatter_motifrmsd(agg, motif).show()
    
df_pos = pd.DataFrame(successes_pos)


In [17]:
df_pos

,motif,task,success_count
0,1bcf,onemotif_twostates_neg,0
1,1prw,onemotif_twostates_neg,0
2,1qjg,onemotif_twostates_neg,1
3,1ycr,onemotif_twostates_neg,8
4,2kl8,onemotif_twostates_neg,6
5,3ixt,onemotif_twostates_neg,32
6,4jhw,onemotif_twostates_neg,0
7,4zyp,onemotif_twostates_neg,5
8,5ius,onemotif_twostates_neg,0
9,5tpn,onemotif_twostates_neg,0


In [18]:
df_neg

,motif,task,success_count
0,1bcf,onemotif_twostates_neg,0
1,1prw,onemotif_twostates_neg,6
2,1qjg,onemotif_twostates_neg,5
3,1ycr,onemotif_twostates_neg,32
4,2kl8,onemotif_twostates_neg,3
5,3ixt,onemotif_twostates_neg,15
6,4jhw,onemotif_twostates_neg,0
7,4zyp,onemotif_twostates_neg,7
8,5ius,onemotif_twostates_neg,0
9,5tpn,onemotif_twostates_neg,0


### two motifs two states

In [14]:
out_dir = f"./out/twomotif_twostates/"
# out_dir = f"./out/twomoitf_twostates_mg/"
# out_dir = f"./out/twomotif_twostates_twoligands/"
# out_dir = "./out/higherantimotif/twomotif_twostates/"

motifA = "3ixt" # active in state 0 when ligand unbound, inactive in state 1 when ligand bound
# motifB = "1ycr" # inactive in state 0 when ligand unbound, active in state 1 when ligand bound
motifB = "6e6r_long" # inactive in state 0 when ligand unbound, active in state 1 when ligand bound

motifA_pdb = f"/data/cb/mihirb14/projects/BoltzDesign1/motifs/{motifA}.pdb"
motifB_pdb = f"/data/cb/mihirb14/projects/BoltzDesign1/motifs/{motifB}.pdb"

motif_out_dir = os.path.join(out_dir,f"{motifA}_{motifB}")
print(motif_out_dir)

./out/twomotif_twostates/3ixt_6e6r_long


In [15]:
motifA_df_full,motifA_df_agg = motif_results(motif_out_dir,motifA_pdb,motifA)
display(plot_scatter_motifrmsd(motifA_df_agg,motifA))
motifB_df_full,motifB_df_agg = motif_results(motif_out_dir,motifB_pdb,motifB)
plot_scatter_motifrmsd(motifB_df_agg,motifB)


In [16]:
success_A = motifA_df_agg[(motifA_df_agg["motifRMSD_mean_unbound"] <= 1.0) & (motifA_df_agg["motifRMSD_mean_bound"] > 1.0)] # motif A active in state 0, inactive in state 1
success_B = motifB_df_agg[(motifB_df_agg["motifRMSD_mean_unbound"] > 1.0) & (motifB_df_agg["motifRMSD_mean_bound"] <= 1.0)] # motif B inactive in state 0, active in state 1
display(success_A)
display(success_B)

,design,motifRMSD_mean_unbound,motifRMSD_mean_bound,motifRMSD_std_unbound,motifRMSD_std_bound,plddt_unbound,plddt_bound,ptm_unbound,ptm_bound
1,design1,0.715412,1.473531,0.243624,0.204120,0.821667,0.638858,0.659451,0.587620
2,design10,0.461271,4.049926,0.034829,0.337558,0.621624,0.591443,0.413206,0.541478
3,design11,0.850377,3.858326,0.190720,1.094986,0.520461,0.499914,0.413122,0.486599
6,design14,0.612373,1.068889,0.071223,0.143678,0.615112,0.468658,0.564180,0.412564
7,design15,0.605893,1.002297,0.063247,0.500533,0.607238,0.673522,0.381231,0.646719
12,design2,0.536641,2.591923,0.054429,0.221172,0.703693,0.604351,0.615407,0.592822
17,design24,0.497459,3.560533,0.058031,0.194014,0.515987,0.457155,0.427259,0.383976
20,design27,0.981371,6.685102,0.634275,0.222055,0.514235,0.513228,0.343852,0.499022
21,design28,0.727089,1.691398,0.098853,0.607777,0.556946,0.534033,0.403548,0.391085
24,design30,0.563938,2.206421,0.172335,1.475327,0.491684,0.400311,0.368592,0.284989


,design,motifRMSD_mean_unbound,motifRMSD_mean_bound,motifRMSD_std_unbound,motifRMSD_std_bound,plddt_unbound,plddt_bound,ptm_unbound,ptm_bound
0,design0,1.724032,0.809588,0.091819,0.093976,0.861940,0.795341,0.835368,0.850010
4,design12,1.227832,0.917158,0.403504,0.543430,0.600917,0.570484,0.481730,0.547607
7,design15,1.279550,0.783273,0.065720,0.055165,0.607238,0.673522,0.381231,0.646719
10,design18,1.804299,0.868677,0.063254,0.041118,0.650403,0.793252,0.424597,0.593477
14,design21,2.083168,0.928924,1.147163,0.104666,0.556811,0.539700,0.495149,0.554767
22,design29,3.148195,0.742699,1.053512,0.098806,0.581364,0.632774,0.457773,0.665205
30,design36,1.215946,0.847333,0.232025,0.203442,0.659256,0.619049,0.537087,0.571193
31,design37,2.188349,0.579647,0.890468,0.047125,0.571974,0.726060,0.385386,0.763578
39,design44,2.163072,0.821756,0.043348,0.075053,0.777332,0.808239,0.682660,0.870640
44,design49,1.253754,0.957988,0.603113,0.122947,0.539274,0.482081,0.410138,0.406018


In [17]:
set(success_A["design"]).intersection(set(success_B["design"]))

{'design15'}

In [58]:
motifB_df_full[motifB_df_full["design"]=="design1"]

,design,state,sample,motifrmsd,plddt,ptm
10,design1,0,0,0.727359,0.679814,0.502158
11,design1,0,1,1.741815,0.710819,0.514460
12,design1,0,2,1.646023,0.709005,0.503220
13,design1,0,3,0.902187,0.684233,0.485717
14,design1,0,4,0.769684,0.672425,0.417673
15,design1,1,0,0.658855,0.767230,0.749349
16,design1,1,1,0.693115,0.755146,0.710810
17,design1,1,2,0.750947,0.776669,0.743204
18,design1,1,3,0.777191,0.758175,0.723047
19,design1,1,4,0.765911,0.774413,0.743548
